<div style="text-align: center;"><img src="https://www.databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning" width="600"></div>



# Demo — Using Feature Engineering in Unity Catalog

In this demo, we will guide you to explore how Feature Engineering in Unity Catalog enhances the feature engineering workflow and understand its crucial role in the development of machine learning models. First we will create feature tables in Unity Catalog for effective implementation in feature engineering processes and then discuss how to update features. Also, we will cover how to convert an existing table to a feature table in Unity Catalog.

**Learning Objectives:**

*By the end of this demo, you will be able to:*

- Create a feature table in Unity Catalog from a PySpark DataFrame with `FeatureEngineeringClient`.
- Identify the requirements for a Delta table in Unity Catalog to be used as a feature table.
- Alter an existing Delta table in Unity Catalog so that it can be used as a feature table.
- Add new features to an existing feature table.
- Remove a feature by dropping its column, with column mapping enabled.
- Read a feature table as it was at an earlier Delta version.
- Explore a feature table on the Features page.
- _(Optional)_: Describe how to upgrade a workspace feature table to a Unity Catalog feature table.


## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #F44336; background: #FFEBEE; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #C62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version ML V6</strong> - <a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976D2; text-decoration: underline;" target="_blank">How to select an environment version</a></li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless ML V6</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## Classroom Setup
Run the following cell to configure your working environment for this course.

This setup will:
- Initialize the `DA` object (Databricks Academy helper)
- Configure your **default catalog** and **schema**
- Provision any supporting configuration needed for this demo

**NOTE:** The `DA` object is only available in Databricks Academy courses.

In [0]:
%run ../Includes/Classroom-Setup-3.2

**Other Conventions:**

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets}")

## Feature Engineering

Before we save features to a feature table we need to create features that we are interested in. Feature selection criteria depend on your project goals and business problem. Thus, in this section, we will pick some features, however, it doesn't necessarily mean that these features are significant for our purpose.

**One important point is that you need to exclude the target field from the feature table and you need to define a primary key for the table.**


### Load Dataset

Typically, first, you will need to conduct data pre-processing and select features. As we covered data pre-processing and feature preparation, we will load a clean dataset which you would typically load from a **`silver`** table.

Let's load in our dataset from a CSV file containing Telco customer churn data from the specified path using Apache Spark. **In this dataset the target column will be `Churn` and primary key will be `customerID`.** A primary key must have no nulls and no duplicates, so the next cell checks this before the feature table is created. The key is used to look up features; it is not itself a model feature.

In [0]:
# Load dataset with spark
shared_volume_name = 'telco' # From Marketplace
csv_name = 'telco-customer-churn' # CSV file name
dataset_path = f"{DA.paths.datasets.telco}/{shared_volume_name}/{csv_name}.csv" # Full path
telco_df = spark.read.csv(dataset_path, header="true", inferSchema="true", multiLine="true", escape='"')

# # Drop the target column
telco_df = telco_df.drop("Churn")

# # Validate the primary key before creating the feature table: no nulls and no duplicates
from pyspark.sql import functions as F
null_keys = telco_df.filter(F.col("customerID").isNull()).count()
duplicate_keys = telco_df.groupBy("customerID").count().filter(F.col("count") > 1).count()
print(f"Null keys: {null_keys}, duplicate keys: {duplicate_keys}")
assert null_keys == 0 and duplicate_keys == 0, "customerID must be a valid primary key"

# # View dataset
display(telco_df)

## Save Features to Feature Table


Let's start creating a <a href="https://docs.databricks.com/aws/en/machine-learning/feature-store/uc/feature-tables-uc" target="_blank">Feature Engineering client</a> so we can populate our feature tables.

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient


fe = FeatureEngineeringClient()


### Create Feature Table

Next, we can create the Feature Table using the **`create_table`** method.

This method takes a few parameters as inputs:
* **`name`** - A feature table name of the form **`<catalog>.<schema_name>.<table_name>`**
* **`primary_keys`** - The primary key(s). If multiple columns are required, specify a list of column names.
* **`timeseries_columns`** - [OPTIONAL] a timestamp column (also part of the primary key) used for point-in-time lookups.
* **`df`** - Data to insert into this feature table. The schema of **`df`** is used as the feature table schema.
* **`schema`** - Feature table schema. Note that either **`schema`** or **`df`** must be provided.
* **`description`** - Description of the feature table
* **`partition_columns`** - Column(s) used to partition the feature table.
* **`tags`** - Tag(s) to tag feature table

In [0]:
# # create a feature table from the dataset
table_name = f"{DA.catalog_name}.{DA.schema_name}.telco_customer_features"

fe.create_table(
    name=table_name,
    primary_keys=["customerID"],
    df=telco_df,
    #partition_columns=["InternetService"] for small datasets partitioning is not recommended
    description="Telco customer features",
    tags={"data_layer": "silver", "source_system": "telco_customer_churn"}
)


Alternatively, you can **`create_table`** with schema only (without **`df`**), and populate data to the feature table with **`fe.write_table`**, **`fe.write_table`** has **`merge`** mode ONLY (to overwrite, we should drop and then re-create the table).


Example:

```
# One time creation
fe.create_table(
    name=table_name,
    primary_keys=["customerID"],
    schema=telco_df.schema,
    description="Original Telco data (Silver)"
)

# Repeated/Scheduled writes
fe.write_table(
    name=table_name,
    df=telco_df,
    mode="merge"
)
```

### Explore Feature Table with the UI

Now let's explore the UI and see how it tracks the tables that we created.

* Click on **Features** from left panel.

* Select the **catalog** that you used for creating the feature table.

* Click on the feature table and you should see the table details as shown below.

![ml-01-feature-store-feature-table-v1](https://files.training.databricks.com/binder/prod_main/data-preparation-for-machine-learning-en_us-4.0.0/images/20261111T171021Z/Data%20Preparation%20for%20Machine%20Learning/Includes/images/ml-01-feature-store-feature-table-v1.png)


### Load Feature Table

We can also look at the metadata of the feature table via the `FeatureEngineeringClient` by using **`get_table()`**. *As a feature table is a Delta table, we can load it with Spark as we normally do for other tables*.

In [0]:
ft = fe.get_table(name=table_name)
print(f"Feature Table description: {ft.description}")
print(ft.features)

In [0]:
display(fe.read_table(name=table_name))


## Update Feature Table

In some cases we might need to update an existing feature table by adding new features or deleting existing features. In this section, we will show how to make these types of changes.

### Add a New Feature

To illustrate adding a new feature, let's redefine an existing one. In this case, we'll transform the `tenure` column by categorizing it into three groups: `short`, `mid`, and `long`, representing different tenure durations. 

Then we will write the dataset back to the feature table. The important parameter is the `mode` parameter, which we should set to `"merge"`.

In [0]:
from pyspark.sql.functions import when

telco_df_updated = telco_df.withColumn("tenure_group", 
    when((telco_df.tenure >= 0) & (telco_df.tenure <= 25), "short")
    .when((telco_df.tenure > 25) & (telco_df.tenure <= 50), "mid")
    .when((telco_df.tenure > 50) & (telco_df.tenure <= 75), "long")
    .otherwise("invalid")
)

Selecting relevant columns. Use an appropriate mode (e.g., "merge") and display the written table for validation.

In [0]:
fe.write_table(
    name=table_name,
    df=telco_df_updated.select("customerID","tenure_group"), # primary_key and column to add
    mode="merge"
)


### Delete Existing Feature

To remove a feature column from the table you can just drop the column. Let's drop the original `tenure` column.

**💡 Note:** Dropping a column requires **column mapping**. Setting `delta.columnMapping.mode = 'name'` enables it and upgrades the table's Delta protocol automatically. Learn more about column mapping ([AWS](https://docs.databricks.com/aws/en/tables/features/column-mapping) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/tables/features/column-mapping) | [GCP](https://docs.databricks.com/gcp/en/tables/features/column-mapping)).

In [0]:
%sql
ALTER TABLE telco_customer_features SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');
ALTER TABLE telco_customer_features DROP COLUMNS (tenure)

## Read Feature Table by Version

As feature tables are based on Delta tables, we get all nice features of Delta including versioning. To demonstrate this, let's read the feature table **as it was just before the `tenure` column was dropped**. Instead of assuming a fixed version number, we look up that version in the table history by its operation.

In [0]:
# # Show the table history, then find the last version before the tenure column was dropped
from pyspark.sql import functions as F

history_df = spark.sql(f"DESCRIBE HISTORY {table_name}").select("version", "timestamp", "operation")
display(history_df.orderBy("version"))

drop_version = history_df.filter(F.col("operation") == "DROP COLUMNS").agg(F.max("version")).first()[0]
before_drop = history_df.filter(F.col("version") == drop_version - 1).first()
timestamp_before_drop = before_drop.timestamp
print(f"Version {before_drop.version} ({before_drop.operation}) at {timestamp_before_drop}")

In [0]:
# # Read previous version using native spark API
telco_df_before_drop = (spark
        .read
        .option("timestampAsOf", timestamp_before_drop)
        .table(table_name))

display(telco_df_before_drop)   # still contains the tenure column

In [0]:
# # Display old version of feature table
feature_df = fe.read_table(
  name=table_name,
  as_of_delta_timestamp=timestamp_before_drop
)

feature_df.printSchema()


## Create a Feature Table from Existing UC Table

Alter/Change existing UC table to become a feature table
Add a primary key (PK) with non-null constraint _(with timestamp if applicable)_ on any UC table to turn it into a feature table. Learn more ([AWS](https://docs.databricks.com/aws/en/machine-learning/feature-store/uc/feature-tables-uc) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/feature-store/uc/feature-tables-uc) | [GCP](https://docs.databricks.com/gcp/en/machine-learning/feature-store/uc/feature-tables-uc))

In this example, the classroom setup (`Classroom-Setup-3.1`) created a Delta table called **`security_features`** in your schema. It contains `customerID` and three security-service columns from the telco data. Let's convert this Delta table to a feature table.

For this, we need to do these two changes;

1. Set primary key columns to `NOT NULL`.

1. Alter the table to add the `Primary Key` Constraint

In [0]:
display(spark.table(f"{DA.catalog_name}.{DA.schema_name}.security_features"))

In [0]:
%sql
ALTER TABLE security_features ALTER COLUMN customerID SET NOT NULL;
ALTER TABLE security_features ADD CONSTRAINT security_features_pk_constraint PRIMARY KEY(customerID);


## _[OPTIONAL]_ Migrate Workspace Feature Table to Unity Catalog

If you have a classic/workspace feature table, you can migrate it to Feature Engineering in Unity Catalog. To do that, first, you will need to upgrade the table to UC supported table and then use `UpgradeClient` to complete the upgrade. For instructions, see Upgrade a workspace feature table to Unity Catalog ([AWS](https://docs.databricks.com/aws/en/machine-learning/feature-store/uc/upgrade-feature-table-to-uc) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/feature-store/uc/upgrade-feature-table-to-uc) | [GCP](https://docs.databricks.com/gcp/en/machine-learning/feature-store/uc/upgrade-feature-table-to-uc)).

A sample code snippet for upgrading classic workspace table;

```
from databricks.feature_engineering import UpgradeClient


upgrade_client = UpgradeClient()

upgrade_client.upgrade_workspace_table(
  source_workspace_table="database.test_features_table",
  target_uc_table=f"{CATALOG}.{SCHEMA}.test_features_table"
)
```


## Conclusion

In this demo, we used Feature Engineering in Unity Catalog to manage features for machine learning. We created feature tables from a PySpark DataFrame, converted an existing Unity Catalog table into a feature table, added and removed features, read a feature table by version, and reviewed how to upgrade a legacy workspace feature table to Unity Catalog. Mastering these skills empowers efficient collaboration and enhances data consistency, contributing to the development of robust machine learning models.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://www.databricks.com/legal/privacynotice" target="_blank">Privacy Notice</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Community</a>

